# 01. Data and Static Baseline

Research question: does a classical, statically-calibrated pairs-trading strategy on the 2024
cryptocurrency spot universe produce a positive risk-adjusted return out-of-sample, once
formation-only calibration, next-bar execution, transaction costs, and a standardized
benchmark-relative evaluation are all held to the standards in `CLAUDE.md`?

This notebook builds the frozen static universe, screens for cointegrated pairs, and evaluates
the two static models: **S0** (the control model, frozen formation parameters, fixed time
stop) and **S1** (S0 + a per-pair half-life-based time stop). Both reuse the exact statistical
and execution engines in `src/stat_arb/` -- no logic is reimplemented here.

**Scope note:** the original research process additionally compared Model S (static) against
two rolling-recalibration variants (RZ, RB) on the 2023 validation period before freezing on
Model S -- a one-time, historical model-selection decision. That comparison is preserved in
`legacy/Stat_Arb_cont_res_1.ipynb` Sections 12-13 for full auditability; this notebook takes its
already-decided outcome (Model S, frozen parameters below) as the given starting point, since
S0 is the control model and is never re-derived or re-edited (`CLAUDE.md`).

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
for _p in (PROJECT_ROOT, PROJECT_ROOT / 'src'):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from config.research_config import (CONFIG, WF_CONFIG, MAX_HALF_LIFE_BARS, MIN_CROSSINGS, MODEL_IDS,
                                     S1_TIME_STOP_MULTIPLIER, S1_TIME_STOP_MULTIPLIER_ROBUSTNESS,
                                     FORWARD_EVAL_START)

## Data loading and universe construction

In [2]:
from stat_arb.data import load_raw_prices, load_raw_returns, filter_universe, split_samples, get_tradable_price

px_raw = load_raw_prices()
ret_raw = load_raw_returns()
print('Raw data:', px_raw.shape, '|', px_raw.index[0], 'to', px_raw.index[-1])

px, ret, dropped = filter_universe(px_raw, ret_raw, CONFIG)
print(f'Excluded ({len(dropped)}):', dropped)
print(f'Remaining universe: {px.shape[1]} coins')

Raw data: (14233, 47) | 2020-01-01 00:00:00 to 2026-06-30 00:00:00


Excluded (19): ['APTUSDT', 'ARBUSDT', 'ENAUSDT', 'FDUSDUSDT', 'FLOKIUSDT', 'IMXUSDT', 'JUPUSDT', 'LDOUSDT', 'OPUSDT', 'PEPEUSDT', 'POLUSDT', 'RENDERUSDT', 'SEIUSDT', 'SUIUSDT', 'TAOUSDT', 'TIAUSDT', 'TONUSDT', 'USDCUSDT', 'XMRUSDT']
Remaining universe: 28 coins


### Coverage and tradability diagnostics

In [3]:
from stat_arb.diagnostics import coverage_table, delisted_coins_report

coverage_table(ret).round(2)

,First Date,Last Date,Observations,Missing (%)
BTCUSDT,2020-01-01 04:00:00,2026-06-30 00:00:00,14230,0.01
ETHUSDT,2020-01-01 04:00:00,2026-06-30 00:00:00,14230,0.01
SOLUSDT,2020-08-11 08:00:00,2026-06-30 00:00:00,12893,0.0
BNBUSDT,2020-01-01 04:00:00,2026-06-30 00:00:00,14230,0.01
XRPUSDT,2020-01-01 04:00:00,2026-06-30 00:00:00,14230,0.01
DOGEUSDT,2020-01-01 04:00:00,2026-06-30 00:00:00,14230,0.01
ADAUSDT,2020-01-01 04:00:00,2026-06-30 00:00:00,14230,0.01
TRXUSDT,2020-01-01 04:00:00,2026-06-30 00:00:00,14230,0.01
AVAXUSDT,2020-09-22 08:00:00,2026-06-30 00:00:00,12641,0.0
SHIBUSDT,2021-05-10 12:00:00,2026-06-30 00:00:00,11260,0.0


In [4]:
delisted = delisted_coins_report(px)
print('Coins with data ending before', px.index[-1].date(), ':')
print(delisted.to_string())

Coins with data ending before 2026-06-30 :
FTMUSDT   2025-01-13
OMUSDT    2026-03-02
MKRUSDT   2025-09-15


### Sample splits

In [5]:
formation_px, validation_px, oos_px = split_samples(px, CONFIG)
print('Formation  :', formation_px.index[0], '-', formation_px.index[-1], f'({len(formation_px)} bars)')
print('Validation :', validation_px.index[0], '-', validation_px.index[-1], f'({len(validation_px)} bars)')
print('Final OOS  :', oos_px.index[0], '-', oos_px.index[-1], f'({len(oos_px)} bars)')
print('\nSplit sanity check passed: no overlap between Formation / Validation / OOS.')

Formation  : 2020-01-01 00:00:00 - 2022-12-31 20:00:00 (6576 bars)
Validation : 2023-01-01 00:00:00 - 2023-12-31 20:00:00 (2190 bars)
Final OOS  : 2024-01-01 00:00:00 - 2026-06-30 00:00:00 (5467 bars)

Split sanity check passed: no overlap between Formation / Validation / OOS.


**Note on the "Final OOS" label.** This range (2024-01-01 to 2026-06-30) is reported
throughout this project as a **Post-Sample Historical Evaluation**, not a pristine, never-seen
holdout: this date range was partially visible in earlier iterations of this research process
before the current engine was finalized, and no parameter in `config/research_config.py` was
changed in response to anything observed in it. It is a corrected historical re-evaluation under
a fixed, pre-specified methodology -- not a forward-looking guarantee. See
`CLAUDE.md`'s "Non-negotiable research principles" #2 and the `FORWARD_EVAL_START` genuinely
untouched window (from 2026-07-01) discussed in notebook 04.

### Asset-Level I(1) Diagnostic

One-time, methodology-documentation check (not a filter): confirms the assets Engle-Granger is
applied to are broadly compatible with the I(1) assumption (non-stationary in log-price level,
stationary in log-returns).

In [6]:
from stat_arb.diagnostics import i1_diagnostic_table

i1_df = i1_diagnostic_table(px)
n_compatible = (i1_df['i1_compatible'] == 'Yes').sum()
print(f'I(1) diagnostic: {n_compatible}/{len(i1_df)} assets are I(1)-compatible.')
i1_df.round(4)

I(1) diagnostic: 24/28 assets are I(1)-compatible.


,asset,level_adf_pvalue,diff_adf_pvalue,i1_compatible,n_obs
0,AAVEUSDT,0.0653,0.0,Yes,12505
1,ADAUSDT,0.1440,0.0,Yes,14232
2,ATOMUSDT,0.7132,0.0,Yes,14232
3,AVAXUSDT,0.1579,0.0,Yes,12642
4,BNBUSDT,0.1503,0.0,Yes,14232
5,BTCUSDT,0.2823,0.0,Yes,14232
6,DOGEUSDT,0.1713,0.0,Yes,14232
7,DOTUSDT,0.9009,0.0,Yes,12848
8,ETHUSDT,0.0417,0.0,No,14232
9,FILUSDT,0.8713,0.0,Yes,12501


## Static pair screen (Section 6 equivalent)

Engle-Granger cointegration (both directions) on every candidate pair -> Benjamini-Hochberg FDR
-> keep the lower-adjusted-p-value direction per pair -> half-life and mean-crossing filters.
No positive-Beta filter here (that is walk-forward-specific, see notebook 02/03).

In [7]:
from stat_arb.pair_selection import run_static_pair_screen

log_formation_all = np.log(formation_px)
candidate_pairs = run_static_pair_screen(log_formation_all, formation_px.columns.tolist(), CONFIG,
                                          MAX_HALF_LIFE_BARS, MIN_CROSSINGS)
print(f'Pairs passing statistical + half-life + crossing screen: {len(candidate_pairs)}')
candidate_pairs[['pair', 'y', 'x', 'pvalue_adj', 'alpha', 'beta', 'half_life_bars', 'crossings']]

Pairs passing statistical + half-life + crossing screen: 12


,pair,y,x,pvalue_adj,alpha,beta,half_life_bars,crossings
0,"(AAVEUSDT, ICPUSDT)",ICPUSDT,AAVEUSDT,1.274069e-07,-5.025595,1.566173,39.210144,144
2,"(FILUSDT, ICPUSDT)",ICPUSDT,FILUSDT,1.444330e-02,0.123612,0.933541,51.517509,154
6,"(GRTUSDT, XLMUSDT)",XLMUSDT,GRTUSDT,1.509586e-02,-1.033310,0.518179,62.266024,187
1,"(ADAUSDT, ICPUSDT)",ICPUSDT,ADAUSDT,7.128640e-04,3.046101,1.471822,63.130345,98
3,"(ICPUSDT, LINKUSDT)",ICPUSDT,LINKUSDT,1.509586e-02,-1.542653,1.688469,75.589382,118
8,"(INJUSDT, XLMUSDT)",XLMUSDT,INJUSDT,2.023596e-02,-2.501311,0.608599,78.383481,165
9,"(AAVEUSDT, XLMUSDT)",AAVEUSDT,XLMUSDT,3.811757e-02,6.966440,1.197289,103.113260,172
10,"(AVAXUSDT, FTMUSDT)",AVAXUSDT,FTMUSDT,4.219162e-02,3.906373,0.675655,177.358951,120
7,"(DOTUSDT, GRTUSDT)",GRTUSDT,DOTUSDT,1.509586e-02,-4.830208,1.395470,216.239113,23
4,"(BTCUSDT, FILUSDT)",BTCUSDT,FILUSDT,1.509586e-02,9.429036,0.305211,227.533213,108


In [8]:
# FROZEN pair list -- do not modify after this cell.
FROZEN_PAIRS = [(row['x'], row['y']) for _, row in candidate_pairs.iterrows()]
print(f'Frozen pair list ({len(FROZEN_PAIRS)} pairs):')
for x, y in FROZEN_PAIRS:
    print(f'  X={x:12s} Y={y}')

Frozen pair list (12 pairs):
  X=AAVEUSDT     Y=ICPUSDT
  X=FILUSDT      Y=ICPUSDT
  X=GRTUSDT      Y=XLMUSDT
  X=ADAUSDT      Y=ICPUSDT
  X=LINKUSDT     Y=ICPUSDT
  X=INJUSDT      Y=XLMUSDT
  X=XLMUSDT      Y=AAVEUSDT
  X=FTMUSDT      Y=AVAXUSDT
  X=DOTUSDT      Y=GRTUSDT
  X=FILUSDT      Y=BTCUSDT
  X=SOLUSDT      Y=ICPUSDT
  X=GRTUSDT      Y=UNIUSDT


## Model freeze (S0)

Model S selected over RZ/RB at 2023 validation (see scope note above). Frozen parameters,
reused unchanged for the entire OOS evaluation.

In [9]:
FROZEN_MODEL = {
    'pairs': FROZEN_PAIRS,
    'model': 'S',
    'rolling_window_days': CONFIG['ROLLING_WINDOW_DAYS'],
    'rolling_window_bars': CONFIG['ROLLING_WINDOW_BARS'],
    'entry_z': CONFIG['ENTRY_Z'],
    'exit_z': CONFIG['EXIT_Z'],
    'hard_stop_z': CONFIG['HARD_STOP_Z'],
    'time_stop_bars': CONFIG['TIME_STOP_BARS'],
    'cost_per_leg_bps': CONFIG['COST_PER_LEG_BPS'],
}
FROZEN_MODEL

{'pairs': [('AAVEUSDT', 'ICPUSDT'),
  ('FILUSDT', 'ICPUSDT'),
  ('GRTUSDT', 'XLMUSDT'),
  ('ADAUSDT', 'ICPUSDT'),
  ('LINKUSDT', 'ICPUSDT'),
  ('INJUSDT', 'XLMUSDT'),
  ('XLMUSDT', 'AAVEUSDT'),
  ('FTMUSDT', 'AVAXUSDT'),
  ('DOTUSDT', 'GRTUSDT'),
  ('FILUSDT', 'BTCUSDT'),
  ('SOLUSDT', 'ICPUSDT'),
  ('GRTUSDT', 'UNIUSDT')],
 'model': 'S',
 'rolling_window_days': 90,
 'rolling_window_bars': 540,
 'entry_z': 1.5,
 'exit_z': 0.5,
 'hard_stop_z': 4.0,
 'time_stop_bars': 156,
 'cost_per_leg_bps': 20}

## S0 -- Out-of-Sample Evaluation

In [10]:
from stat_arb.execution import run_model_on_pair

px_oos_engine = px.loc[:CONFIG['OOS_END']]
ret_oos_engine = ret.loc[:CONFIG['OOS_END']]

oos_results, oos_trades = {}, {}
for x_name, y_name in FROZEN_MODEL['pairs']:
    result, trades = run_model_on_pair(
        x_name, y_name, FROZEN_MODEL['model'], FROZEN_MODEL['rolling_window_bars'],
        FROZEN_MODEL['entry_z'], FROZEN_MODEL['exit_z'], FROZEN_MODEL['hard_stop_z'],
        FROZEN_MODEL['time_stop_bars'], FROZEN_MODEL['cost_per_leg_bps'],
        px_oos_engine, ret_oos_engine, CONFIG, state_start=CONFIG['OOS_START'])
    result = result.loc[CONFIG['OOS_START']:CONFIG['OOS_END']]
    assert result[['qty_x', 'qty_y']].iloc[0].abs().sum() == 0.0
    assert result[['qty_x', 'qty_y']].iloc[-1].abs().sum() == 0.0
    oos_results[(x_name, y_name)] = result
    oos_trades[(x_name, y_name)] = trades

print(f'S0 OOS run complete on {len(oos_results)} pairs. FLAT-start / flat-end checks PASSED.')

S0 OOS run complete on 12 pairs. FLAT-start / flat-end checks PASSED.


In [11]:
from stat_arb.reporting import standardized_model_summary

S0_ROW, S0_OOS_PORTFOLIO = standardized_model_summary(
    'S0_STATIC_BASELINE', oos_results, FROZEN_MODEL['pairs'], px_oos_engine,
    FROZEN_MODEL['cost_per_leg_bps'], CONFIG['OOS_START'], CONFIG['OOS_END'], CONFIG['BARS_PER_YEAR'],
    trades_dict=oos_trades)
assert S0_OOS_PORTFOLIO['coin_qty'].iloc[-1].abs().max() < 1e-12
pd.Series(S0_ROW)

model                      S0_STATIC_BASELINE
daily_sharpe_252                    -0.115524
bar_sharpe_secondary                 -0.15277
ann_volatility                       0.038673
cum_net_return                      -0.016482
cum_gross_return                    -0.007201
max_drawdown                        -0.111707
n_trades                                   29
n_pairs_traded                             11
win_rate                             0.482759
avg_trade_return                    -0.002386
median_trade_return                 -0.002345
avg_holding_bars                   125.862069
median_holding_bars                     156.0
time_in_market_pct                   0.422718
turnover                             4.728719
pct_exit_mean_reversion              0.241379
pct_exit_time_stop                   0.655172
pct_exit_hard_stop                   0.068966
total_costs_pct                      0.009281
btc_correlation                      0.023452
btc_beta                          

## S1 -- Static Model with Pair-Specific Half-Life Time Stops

In [12]:
from stat_arb.pair_selection import compute_pair_time_stops
from stat_arb.execution import run_s1_universe

s1_time_stops, s1_ineligible = compute_pair_time_stops(FROZEN_PAIRS, formation_px, S1_TIME_STOP_MULTIPLIER)
S1_ELIGIBLE_PAIRS = [p for p in FROZEN_PAIRS if p in s1_time_stops]

px_oos_engine_s1 = px.loc[:CONFIG['OOS_END']]
ret_oos_engine_s1 = ret.loc[:CONFIG['OOS_END']]
S1_OOS_RESULTS, S1_OOS_TRADES = run_s1_universe(
    S1_ELIGIBLE_PAIRS, s1_time_stops, px_oos_engine_s1, ret_oos_engine_s1, CONFIG, CONFIG['OOS_START'])
S1_OOS_RESULTS = {k: v.loc[CONFIG['OOS_START']:CONFIG['OOS_END']] for k, v in S1_OOS_RESULTS.items()}

for (x, y), result in S1_OOS_RESULTS.items():
    assert result[['qty_x', 'qty_y']].iloc[0].abs().sum() == 0.0
    assert result[['qty_x', 'qty_y']].iloc[-1].abs().sum() == 0.0
print(f'S1 OOS run complete on {len(S1_OOS_RESULTS)} pairs. FLAT-start / flat-end checks PASSED.')
if s1_ineligible:
    print('Ineligible for S1:', s1_ineligible)

S1 OOS run complete on 12 pairs. FLAT-start / flat-end checks PASSED.


In [13]:
S1_ROW, S1_OOS_PORTFOLIO = standardized_model_summary(
    'S1_STATIC_PAIR_HALFLIFE_STOP', S1_OOS_RESULTS, S1_ELIGIBLE_PAIRS, px_oos_engine_s1,
    CONFIG['COST_PER_LEG_BPS'], CONFIG['OOS_START'], CONFIG['OOS_END'], CONFIG['BARS_PER_YEAR'],
    trades_dict=S1_OOS_TRADES)

s0_vs_s1_df = pd.DataFrame([S0_ROW, S1_ROW]).set_index('model')
print('S0 vs. S1 -- OOS (2024-01-01 to 2026-06-30), standardized comparison:')
s0_vs_s1_df.round(4).T

S0 vs. S1 -- OOS (2024-01-01 to 2026-06-30), standardized comparison:


model,S0_STATIC_BASELINE,S1_STATIC_PAIR_HALFLIFE_STOP
daily_sharpe_252,-0.1155,-0.1249
bar_sharpe_secondary,-0.1528,-0.1613
ann_volatility,0.0387,0.0416
cum_net_return,-0.0165,-0.0188
cum_gross_return,-0.0072,-0.0092
max_drawdown,-0.1117,-0.1053
n_trades,29.0000,29.0000
n_pairs_traded,11.0000,11.0000
win_rate,0.4828,0.5517
avg_trade_return,-0.0024,0.0013


### S1 time-stop multiplier robustness (2.0x / 2.5x primary / 3.0x)

Pre-specified sensitivity grid -- no grid search, no selection on OOS performance. Instability
across nearby multipliers is itself a reportable finding, not a prompt to adopt the
best-performing variant.

In [14]:
S1_MULTIPLIER_ROWS = []
for mult in S1_TIME_STOP_MULTIPLIER_ROBUSTNESS:
    ts_mult, ineligible_mult = compute_pair_time_stops(FROZEN_PAIRS, formation_px, mult)
    eligible_mult = [p for p in FROZEN_PAIRS if p in ts_mult]
    results_mult, trades_mult = run_s1_universe(eligible_mult, ts_mult, px_oos_engine_s1, ret_oos_engine_s1, CONFIG, CONFIG['OOS_START'])
    results_mult = {k: v.loc[CONFIG['OOS_START']:CONFIG['OOS_END']] for k, v in results_mult.items()}
    row, _ = standardized_model_summary(
        f'S1_x{mult}', results_mult, eligible_mult, px_oos_engine_s1, CONFIG['COST_PER_LEG_BPS'],
        CONFIG['OOS_START'], CONFIG['OOS_END'], CONFIG['BARS_PER_YEAR'])
    S1_MULTIPLIER_ROWS.append(row)

pd.DataFrame(S1_MULTIPLIER_ROWS).set_index('model')[['daily_sharpe_252', 'cum_net_return', 'max_drawdown']].round(4)

,daily_sharpe_252,cum_net_return,max_drawdown
model,,,
S1_x2.0,0.3425,0.0413,-0.0853
S1_x2.5,-0.1249,-0.0188,-0.1053
S1_x3.0,-0.2010,-0.0315,-0.1113


## Interpretation

S0 and S1 OOS Sharpe and cumulative return are both negative on this sample (see table above) --
a negative result is an acceptable, reportable outcome per `CLAUDE.md`. The S1 multiplier
robustness sweep shows Sharpe swinging materially across 2.0x/2.5x/3.0x, meaning the time-stop
choice does not robustly explain S0's underperformance; this instability is itself the finding,
not a basis for picking the best-performing variant.

In [15]:
s0_s1_tables_dir = PROJECT_ROOT / 'results' / 'tables'
s0_s1_tables_dir.mkdir(parents=True, exist_ok=True)
with open(s0_s1_tables_dir / 's0_s1_rows.json', 'w') as f:
    json.dump({'S0_STATIC_BASELINE': S0_ROW, 'S1_STATIC_PAIR_HALFLIFE_STOP': S1_ROW}, f, indent=2, default=str)
print(f'Saved S0/S1 rows to {s0_s1_tables_dir / "s0_s1_rows.json"}')

Saved S0/S1 rows to C:\Users\kuhfu\OneDrive\A_Nils\Studium\Quant\WSQ\My Project\Stat Arb - Pair Trading\results\tables\s0_s1_rows.json
